In [ ]:
# Algoritmo genético para el TSP
Nombre: Pedro Ortega Soldevila <br>
Link: https://colab.research.google.com/drive/1FmGCEUP8SU37UmckRWFmsraiymi7Wq42?usp=sharing <br>
Github: https://github.com/portega02/Algoritmos-de-optimizaci-n/edit/main/Algoritmo_Genetico_para_el_TSP.ipynb

# Instalación de librerias

In [1]:
!pip install requests
import urllib.request

!pip install tsplib95
import tsplib95

In [2]:
import random   #Libreria para generar numeros y listas aleatorias
import copy     #Permite hacer copias de objetos en python: listas, diccionarios,...

# Carga de datos del problema

In [4]:
#Librerias y carga del problema

#http://elib.zib.de/pub/mp-testdata/tsp/tsplib/
#Documentacion :
  # https://wwwproxy.iwr.uni-heidelberg.de/groups/comopt/software/TSPLIB95/tsp95.pdf
  # https://tsplib95.readthedocs.io/usage.html
  # https://tsplib95.readthedocs.io/modules.html#module-tsplib95.models

#Matriz de distancias
# file = "swiss42.tsp" ;
# urllib.request.urlretrieve("http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/swiss42.tsp.gz", file + '.gz')
# !gzip -d swiss42.tsp.gz     #Descomprimir el fichero de datos

file = "swiss42.tsp"
url = "https://raw.githubusercontent.com/coin-or/jorlib/master/jorlib-core/src/test/resources/tspLib/tsp/swiss42.tsp"

urllib.request.urlretrieve(url, file)

#Objeto de tsplib95 para nuestro problema problema
problem = tsplib95.load(file)

#Nodos
Nodos = list(problem.get_nodes())

#Aristas
Aristas = list(problem.get_edges())
print(Aristas)

#Coordenadas(si estan disponibles en el ficher)
problem.get_display(1)

#Distancia
problem.get_weight(1, 2)



[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9), (0, 10), (0, 11), (0, 12), (0, 13), (0, 14), (0, 15), (0, 16), (0, 17), (0, 18), (0, 19), (0, 20), (0, 21), (0, 22), (0, 23), (0, 24), (0, 25), (0, 26), (0, 27), (0, 28), (0, 29), (0, 30), (0, 31), (0, 32), (0, 33), (0, 34), (0, 35), (0, 36), (0, 37), (0, 38), (0, 39), (0, 40), (0, 41), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (1, 16), (1, 17), (1, 18), (1, 19), (1, 20), (1, 21), (1, 22), (1, 23), (1, 24), (1, 25), (1, 26), (1, 27), (1, 28), (1, 29), (1, 30), (1, 31), (1, 32), (1, 33), (1, 34), (1, 35), (1, 36), (1, 37), (1, 38), (1, 39), (1, 40), (1, 41), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (2, 9), (2, 10), (2, 11), (2, 12), (2, 13), (2, 14), (2, 15), (2, 16), (2, 17), (2, 18), (2, 19), (2, 20), (2, 21), (2, 22), (2, 23), (2, 24), (2, 25), (2, 26), (2, 27), (2, 28), (2, 29), (2,

34

# Funciones de la Actividad Guiada 3

In [5]:
#Funciones de la Actividad Guiada 3
#Se genera una solucion aleatoria con comienzo en en el nodo 0
def crear_solucion(Nodos):
  solucion = [Nodos[0]]
  for n in Nodos[1:] :
    solucion = solucion + [random.choice(list(set(Nodos) - set(solucion)))]
  return solucion

#Devuelve la distancia entre dos nodos
def distancia(a,b, problem):
  return problem.get_weight(a,b)


#Devuelve la distancia total de una trayectoria/solucion
def distancia_total(solucion, problem):
  distancia_total = 0
  for i in range(len(solucion)-1):
    distancia_total += distancia(solucion[i] ,solucion[i+1] ,  problem)
  return distancia_total + distancia(solucion[len(solucion)-1] ,solucion[0], problem)

# Funciones Auxiliares

In [6]:
#Genera una poblacion inicial de soluciones de tamaño N.
# Puede ser válida la solución aleatoria de la AG3: crear_solucion(Nodos)
def generar_poblacion(Nodos, N):
    poblacion = []
    
    for _ in range(N):
        poblacion.append(crear_solucion(Nodos))
    
    return poblacion

In [7]:
#Evalua la población y devuelve el mejor individuo
def Evaluar_Poblacion(poblacion, problem):
    mejor_solucion = None
    mejor_distancia = float("inf")
    
    for solucion in poblacion:
        distancia = distancia_total(solucion, problem)
        
        if distancia < mejor_distancia:
            mejor_solucion = solucion
            mejor_distancia = distancia
    
    return mejor_solucion, mejor_distancia

In [8]:
#Funcion de cruce. Recibe una poblacion(lista de soluciones) y devuelve la población ampliada con los hijos.
# Todos los individuos de la población son selecionados para el cruce(si la población es par)
# Podría aplicarse un proceso previo de selección para elegir los individuos que se desea cruzar.
def Cruzar(poblacion, mutacion, problem):
    nueva_poblacion = poblacion.copy()
    
    random.shuffle(poblacion)
    
    for i in range(0, len(poblacion)-1, 2):
        padres = [poblacion[i], poblacion[i+1]]
        hijo1, hijo2 = Descendencia(padres, problem, mutacion)
        
        nueva_poblacion.append(hijo1)
        nueva_poblacion.append(hijo2)
    
    return nueva_poblacion

In [9]:
#Funcion para generar hijos a partir de 2 padres:
# Se elige el metodo de 1-punto de corte pero es posible usar otros n-puntos, uniforme, dependiendo del problema
def Descendencia(padres, problem, mutacion):
    padre1 = padres[0]
    padre2 = padres[1]
    
    punto_corte = random.randint(1, len(padre1)-2)
    
    hijo1 = padre1[:punto_corte] + padre2[punto_corte:]
    hijo2 = padre2[:punto_corte] + padre1[punto_corte:]
    
    hijo1 = Factibilizar(hijo1, problem)
    hijo2 = Factibilizar(hijo2, problem)
    
    hijo1 = Mutar(hijo1, mutacion)
    hijo2 = Mutar(hijo2, mutacion)
    
    return hijo1, hijo2

In [10]:
#Para el operador de cruce 1-punto los hijos generados no son soluciones(algunos nodos se repiten y otros no están)
def Factibilizar(solucion, problem):
    Nodos = list(problem.get_nodes())
    
    faltan = list(set(Nodos) - set(solucion))
    vistos = set()
    solucion_factible = solucion.copy()
    
    for i in range(len(solucion_factible)):
        if solucion_factible[i] in vistos:
            solucion_factible[i] = faltan.pop(0)
        else:
            vistos.add(solucion_factible[i])
    
    return solucion_factible

In [11]:
#Funcion de mutación. Se eligen dos nodos y se intercambia. Se podrian añadir otros operaradores
# Se hace mutaciones mutacion% de las veces
def Mutar(solucion, mutacion):
    nueva_solucion = solucion.copy()
    
    if random.random() < mutacion:
        i, j = random.sample(range(1, len(nueva_solucion)), 2)
        nueva_solucion[i], nueva_solucion[j] = nueva_solucion[j], nueva_solucion[i]
    
    return nueva_solucion

In [12]:
#Funcion de seleccion de la población. Recibe como parametro una poblacion y
# devuelve una poblacion a la que se ha eliminado individuos poco aptos(fitness alto) y para mantener una poblacion estable de N individuos
#Se tiene en cuenta el porcentaje elitismo pasado como parametro
# Para los individuos que no son de la elite podríamos usar una selección de ruleta(proporcional a su fitness)
def Seleccionar(problem, poblacion, N, elitismo):
    poblacion_ordenada = sorted(poblacion, key=lambda x: distancia_total(x, problem))
    
    n_elite = int(elitismo)
    elite = poblacion_ordenada[:n_elite]
    
    resto = poblacion_ordenada[n_elite:]
    random.shuffle(resto)
    
    seleccionados = elite + resto[:N - n_elite]
    
    return seleccionados


In [13]:
poblacion_prueba = generar_poblacion(Nodos, 10)
mejor_sol_prueba, mejor_dist_prueba = Evaluar_Poblacion(poblacion_prueba, problem)

print("Tamaño de la población:", len(poblacion_prueba))
print("Mejor solución encontrada en la población de prueba:")
print(mejor_sol_prueba)
print("Mejor distancia:", mejor_dist_prueba)

Tamaño de la población: 10
Mejor solución encontrada en la población de prueba:
[0, 11, 2, 30, 41, 26, 29, 19, 17, 15, 13, 5, 14, 28, 22, 40, 10, 35, 8, 24, 4, 7, 18, 9, 1, 37, 31, 27, 33, 32, 12, 20, 21, 6, 36, 16, 25, 23, 38, 39, 3, 34]
Mejor distancia: 4483


# Proceso Principal

In [14]:
#Funcion principal del algoritmo genetico
#######################################################3
def algoritmo_genetico(problem=problem,N=100,mutacion=.15,elitismo=.1,generaciones=100):
  # problem = datos del problema
  # N = Tamaño de la población
  # mutacion = probabilidad de una mutación
  # elitismo = porcion de la mejor poblacion a mantener
  # generaciones = nº de generaciones a generar para finalizar

  #Genera la poblacion inicial
  Nodos = list(problem.get_nodes())
  poblacion = generar_poblacion(Nodos,N)

  #Inicializamos valores para la mejor solucion
  (mejor_solucion, mejor_distancia) = Evaluar_Poblacion(poblacion, problem)


  #Condicion de parada
  parar = False
  n=0
  #Inciamos el cliclo de generaciones
  while(parar == False) :

    #Cruce de la poblacion(incluye mutación)
    poblacion = Cruzar(poblacion, mutacion, problem)

    #Seleccionamos la población
    poblacion = Seleccionar(problem,poblacion, N, elitismo)

    #Evaluamos la nueva población
    (mejor_solucion, mejor_distancia) = Evaluar_Poblacion(poblacion, problem)

    print("Generacion #", n, "\nLa mejor solución es:" , mejor_solucion, "\ncon distancia " , mejor_distancia, "\n")

    #Numero de generaciones. Criterio de parada
    if n==generaciones:
      parar = True
    n +=1

  return mejor_solucion


sol = algoritmo_genetico(problem=problem,N=500,mutacion=.3,elitismo=.40,generaciones=250)

Generacion # 0 
La mejor solución es: [0, 3, 37, 11, 41, 21, 23, 24, 38, 30, 20, 18, 28, 6, 1, 7, 19, 27, 26, 8, 9, 32, 15, 16, 36, 2, 40, 13, 34, 33, 25, 4, 5, 39, 22, 10, 14, 29, 31, 17, 35, 12] 
con distancia  3986 

Generacion # 1 
La mejor solución es: [0, 6, 32, 35, 13, 26, 9, 1, 28, 10, 21, 39, 24, 25, 4, 41, 40, 12, 7, 19, 8, 31, 34, 17, 23, 38, 33, 2, 20, 29, 3, 11, 22, 30, 36, 37, 5, 14, 15, 27, 16, 18] 
con distancia  4062 

Generacion # 2 
La mejor solución es: [0, 37, 35, 36, 10, 20, 1, 11, 8, 32, 29, 33, 24, 25, 26, 5, 15, 23, 21, 2, 9, 6, 17, 18, 7, 31, 34, 3, 4, 39, 40, 41, 12, 13, 14, 16, 19, 28, 22, 27, 30, 38] 
con distancia  3808 

Generacion # 3 
La mejor solución es: [0, 37, 35, 36, 10, 20, 1, 11, 8, 32, 29, 33, 24, 25, 26, 5, 15, 23, 21, 2, 9, 6, 17, 18, 7, 31, 34, 3, 4, 39, 40, 41, 12, 13, 14, 16, 19, 28, 22, 27, 30, 38] 
con distancia  3808 

Generacion # 4 
La mejor solución es: [0, 16, 35, 41, 22, 30, 20, 11, 5, 6, 38, 29, 32, 3, 31, 28, 4, 39, 9, 23, 8, 14, 

### Proceso principal del algoritmo genético

El algoritmo genético parte de una población inicial de soluciones y la mejora a lo largo de varias generaciones.

En cada iteración se aplican operadores de cruce y mutación, y posteriormente se seleccionan los individuos más aptos para formar la nueva población. De este modo, la población evoluciona progresivamente hacia soluciones de menor distancia total.

In [15]:
##Funcion principal del algoritmo genético
##############################################################
def algoritmo_genetico(problem=problem, N=100, mutacion=0.15, elitismo=10, generaciones=50):
    # problem = datos del problema
    # N = Tamaño de la población
    # mutacion = probabilidad de mutación
    # elitismo = numero de individuos de la mejor población a mantener
    # generaciones = nº de generaciones a generar para finalizar

    #Genera la población inicial
    Nodos = list(problem.get_nodes())
    poblacion = generar_poblacion(Nodos, N)

    #Inicializamos valores para la mejor solución
    mejor_solucion, mejor_distancia = Evaluar_Poblacion(poblacion, problem)

    #Condicion de parada
    parar = False
    n = 0

    #Iniciamos el ciclo de generaciones
    while parar == False:

        #Cruce de la población (incluye mutación)
        poblacion = Cruzar(poblacion, mutacion, problem)

        #Seleccionamos la población
        poblacion = Seleccionar(problem, poblacion, N, elitismo)

        #Evaluamos la nueva población
        mejor_solucion, mejor_distancia = Evaluar_Poblacion(poblacion, problem)

        print("Generacion #", n, "\nLa mejor solución es:", mejor_solucion, "\ncon distancia:", mejor_distancia)

        #Numero de generaciones. Criterio de parada
        if n == generaciones:
            parar = True

        n += 1

    return mejor_solucion

sol_gen = algoritmo_genetico(problem=problem, N=100, mutacion=0.3, elitismo=10, generaciones=20)

print("Solución final:")
print(sol_gen)
print("Distancia final:", distancia_total(sol_gen, problem))

Generacion # 0 
La mejor solución es: [0, 15, 1, 30, 34, 29, 21, 4, 27, 23, 6, 3, 2, 5, 13, 19, 11, 31, 41, 36, 33, 32, 20, 35, 25, 18, 17, 8, 28, 9, 22, 24, 10, 26, 12, 40, 38, 39, 37, 7, 14, 16] 
con distancia: 3924
Generacion # 1 
La mejor solución es: [0, 30, 24, 38, 31, 13, 16, 15, 37, 19, 2, 6, 25, 40, 21, 10, 29, 28, 12, 7, 1, 35, 5, 8, 34, 14, 32, 33, 3, 22, 4, 9, 41, 39, 36, 17, 20, 18, 11, 26, 23, 27] 
con distancia: 3872
Generacion # 2 
La mejor solución es: [0, 30, 24, 38, 31, 13, 16, 15, 37, 19, 2, 6, 25, 40, 21, 10, 29, 28, 12, 7, 1, 35, 5, 8, 34, 14, 32, 33, 3, 22, 4, 9, 41, 39, 36, 17, 20, 18, 11, 26, 23, 27] 
con distancia: 3872
Generacion # 3 
La mejor solución es: [0, 30, 24, 38, 31, 13, 16, 15, 37, 19, 2, 6, 25, 40, 21, 10, 29, 28, 12, 7, 1, 35, 5, 8, 34, 14, 32, 33, 3, 22, 4, 9, 41, 39, 36, 17, 20, 18, 11, 26, 23, 27] 
con distancia: 3872
Generacion # 4 
La mejor solución es: [0, 30, 24, 38, 31, 13, 16, 15, 37, 19, 2, 20, 25, 40, 21, 10, 29, 28, 12, 7, 1, 35, 5, 33